# 🔬 Skin Cancer Detection — Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/skin-cancer-eda/blob/main/Skin_Cancer_ISIC_EDA.ipynb)

---

**Dataset:** ISIC — International Skin Imaging Collaboration  
**Total Images:** 2,357  
**Classes:** 9 oncological disease categories  
**License:** CC0 Public Domain  

---

### 📌 About This Notebook

Melanoma accounts for **75% of all skin cancer deaths**. Early detection through automated image analysis can significantly reduce the manual effort required for diagnosis.

This notebook performs a full **Exploratory Data Analysis (EDA)** on the ISIC skin cancer dataset covering:

- ✅ Dataset structure and folder exploration
- ✅ Class distribution and image counts
- ✅ Train / Test split analysis
- ✅ Class imbalance detection
- ✅ Sample image visualization per class
- ✅ Summary statistics

---

### 🏷️ Disease Classes

| # | Disease | Type |
|---|---------|------|
| 1 | Melanoma | Malignant |
| 2 | Nevus | Benign |
| 3 | Pigmented Benign Keratosis | Benign |
| 4 | Basal Cell Carcinoma | Malignant |
| 5 | Actinic Keratosis | Pre-malignant |
| 6 | Seborrheic Keratosis | Benign |
| 7 | Squamous Cell Carcinoma | Malignant |
| 8 | Dermatofibroma | Benign |
| 9 | Vascular Lesion | Benign/Malignant |

---
## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries (Pillow is pre-installed in Colab but included for safety)
!pip install Pillow -q

# ── Standard Libraries ─────────────────────────────────────────────────────────
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

# ── Data Handling ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

# ── Image Processing ───────────────────────────────────────────────────────────
from PIL import Image

print('✅ All libraries imported successfully!')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')
print(f'   Pillow  : {Image.__version__}')

---
## 📂 Step 2 — Upload & Extract Dataset

Upload your dataset ZIP file when prompted.  
The ZIP should contain a `Train/` and `Test/` folder, each with 9 class sub-folders.

**Expected structure:**
```
dataset.zip
├── Train/
│   ├── Melanoma/
│   ├── Nevus/
│   └── ... (9 folders total)
└── Test/
    ├── Melanoma/
    ├── Nevus/
    └── ... (9 folders total)
```

In [ ]:
from google.colab import files

print('📂 Please upload your dataset ZIP file...')
uploaded = files.upload()

# Get the uploaded filename
zip_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_filename}')
print(f'📦 Extracting to /content/skin_cancer/ ...')

# Extract ZIP
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall('/content/skin_cancer')

print('\n✅ Extraction complete!')
print('\n📁 Folder Structure:')
print('=' * 40)

# Display folder tree (2 levels deep)
for root, dirs, files_list in os.walk('/content/skin_cancer'):
    level = root.replace('/content/skin_cancer', '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    if level == 2:
        print(f'{indent}  └── {len(files_list)} images')

---
## 🔍 Step 3 — Scan Dataset & Build Summary

Automatically detects the `Train/` and `Test/` folders and counts images per class.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
BASE_PATH      = '/content/skin_cancer'
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# ── Auto-detect base path (handles nested ZIPs) ───────────────────────────────
for root, dirs, _ in os.walk(BASE_PATH):
    if 'Train' in dirs or 'train' in dirs:
        BASE_PATH = root
        break

TRAIN_PATH = os.path.join(BASE_PATH, 'Train') if os.path.exists(os.path.join(BASE_PATH, 'Train')) \
             else os.path.join(BASE_PATH, 'train')
TEST_PATH  = os.path.join(BASE_PATH, 'Test')  if os.path.exists(os.path.join(BASE_PATH, 'Test'))  \
             else os.path.join(BASE_PATH, 'test')

print(f'📁 Train path : {TRAIN_PATH}')
print(f'📁 Test path  : {TEST_PATH}')

# ── Count images per class ────────────────────────────────────────────────────
def count_images_per_class(folder_path):
    """Count image files inside each class sub-folder."""
    class_counts = {}
    if not os.path.exists(folder_path):
        print(f'⚠️  Path not found: {folder_path}')
        return class_counts
    for class_name in sorted(os.listdir(folder_path)):
        class_dir = os.path.join(folder_path, class_name)
        if os.path.isdir(class_dir):
            imgs = [
                f for f in os.listdir(class_dir)
                if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS
            ]
            class_counts[class_name] = len(imgs)
    return class_counts

train_counts = count_images_per_class(TRAIN_PATH)
test_counts  = count_images_per_class(TEST_PATH)

# ── Build Summary DataFrame ───────────────────────────────────────────────────
all_classes = sorted(set(list(train_counts.keys()) + list(test_counts.keys())))

df_summary = pd.DataFrame({
    'Class'  : all_classes,
    'Train'  : [train_counts.get(c, 0) for c in all_classes],
    'Test'   : [test_counts.get(c, 0)  for c in all_classes],
})
df_summary['Total']   = df_summary['Train'] + df_summary['Test']
df_summary['Train %'] = (df_summary['Train'] / df_summary['Total'] * 100).round(1)
df_summary['Test %']  = (df_summary['Test']  / df_summary['Total'] * 100).round(1)
df_summary            = df_summary.sort_values('Total', ascending=False).reset_index(drop=True)

# ── Display ───────────────────────────────────────────────────────────────────
print('\n📊 Dataset Summary')
print('=' * 65)
print(df_summary.to_string(index=False))
print('=' * 65)
print(f'  🖼️  Total images  : {df_summary["Total"].sum():,}')
print(f'  📁  Total classes : {len(all_classes)}')
print(f'  🏋️  Train images  : {df_summary["Train"].sum():,}')
print(f'  🧪  Test images   : {df_summary["Test"].sum():,}')
print(f'  ⚖️  Imbalance     : {df_summary["Total"].max() / df_summary["Total"].min():.1f}x '
      f'({df_summary.iloc[0]["Class"]} vs {df_summary.iloc[-1]["Class"]})')
print('=' * 65)

---
## 📊 Step 4 — Class Distribution Analysis

In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────────
COLORS = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C', '#CCB974'
]
class_colors = {cls: COLORS[i % len(COLORS)]
                for i, cls in enumerate(df_summary['Class'])}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Class Distribution Analysis', fontsize=15,
             fontweight='bold', color='#1a1a2e', y=1.02)

# Bar chart
ax = axes[0]
bars = ax.bar(
    df_summary['Class'], df_summary['Total'],
    color=[class_colors[c] for c in df_summary['Class']],
    edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, df_summary['Total']):
    pct = val / df_summary['Total'].sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5,
            f'{val}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=8, color='#333')
ax.set_title('Total Images per Class', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Disease Class', fontsize=10)
ax.set_ylabel('Number of Images', fontsize=10)
ax.set_xticklabels(df_summary['Class'], rotation=35, ha='right', fontsize=8)
ax.set_ylim(0, df_summary['Total'].max() * 1.22)
ax.set_facecolor('white')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Pie chart
ax2 = axes[1]
wedges, _, autotexts = ax2.pie(
    df_summary['Total'],
    labels=None,
    autopct='%1.1f%%',
    colors=[class_colors[c] for c in df_summary['Class']],
    startangle=140,
    pctdistance=0.80,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(8)
ax2.set_title('Class Share (%)', fontsize=12, fontweight='bold', pad=12)
ax2.legend(
    df_summary['Class'], loc='lower center',
    bbox_to_anchor=(0.5, -0.18), ncol=3,
    fontsize=7.5, frameon=False
)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution plot saved as class_distribution.png')

---
## 🔀 Step 5 — Train / Test Split Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Train / Test Split Analysis', fontsize=15,
             fontweight='bold', color='#1a1a2e', y=1.02)

# ── Grouped bar: Train vs Test per class ──────────────────────────────────────
ax = axes[0]
x     = np.arange(len(df_summary))
width = 0.38
b1 = ax.bar(x - width/2, df_summary['Train'], width,
            label='Train', color='#4C72B0', edgecolor='white', linewidth=0.7)
b2 = ax.bar(x + width/2, df_summary['Test'],  width,
            label='Test',  color='#DD8452', edgecolor='white', linewidth=0.7)
for bar, val in zip(b1, df_summary['Train']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', va='bottom', fontsize=7.5, color='#333')
for bar, val in zip(b2, df_summary['Test']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', va='bottom', fontsize=7.5, color='#333')
ax.set_xticks(x)
ax.set_xticklabels(df_summary['Class'], rotation=35, ha='right', fontsize=8)
ax.set_title('Train vs Test Count per Class', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Number of Images', fontsize=10)
ax.legend(fontsize=10, frameon=False)
ax.set_facecolor('white')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
ax.set_ylim(0, max(df_summary['Train'].max(), df_summary['Test'].max()) * 1.18)

# ── Stacked percentage bar ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.barh(df_summary['Class'], df_summary['Train %'],
         color='#4C72B0', label='Train %', height=0.55)
ax2.barh(df_summary['Class'], df_summary['Test %'],
         left=df_summary['Train %'],
         color='#DD8452', label='Test %', height=0.55)
for i, row in df_summary.iterrows():
    ax2.text(row['Train %']/2, i, f"{row['Train %']}%",
             ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    ax2.text(row['Train %'] + row['Test %']/2, i, f"{row['Test %']}%",
             ha='center', va='center', fontsize=8, color='white', fontweight='bold')
ax2.set_title('Train / Test Split % per Class', fontsize=12, fontweight='bold', pad=12)
ax2.set_xlabel('Percentage (%)', fontsize=10)
ax2.legend(fontsize=10, frameon=False, loc='lower right')
ax2.set_xlim(0, 110)
ax2.set_facecolor('white')
ax2.grid(axis='x', alpha=0.3)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig('train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Train/Test split plot saved as train_test_split.png')

---
## ⚖️ Step 6 — Class Imbalance Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#f8f9fa')

# Sort ascending for horizontal bar
sorted_df  = df_summary.sort_values('Total')
mean_count = df_summary['Total'].mean()

bars = ax.barh(
    sorted_df['Class'], sorted_df['Total'],
    color=[class_colors[c] for c in sorted_df['Class']],
    edgecolor='white', linewidth=0.7, height=0.6
)

# Value labels
for bar, val in zip(bars, sorted_df['Total']):
    pct = val / df_summary['Total'].sum() * 100
    ax.text(val + 3, bar.get_y() + bar.get_height()/2,
            f'{val}  ({pct:.1f}%)',
            va='center', fontsize=9, color='#333')

# Mean line
ax.axvline(mean_count, color='red', linewidth=1.8,
           linestyle='--', zorder=5, label=f'Mean = {mean_count:.0f}')
ax.text(mean_count + 3, len(sorted_df) - 0.3,
        f'Mean: {mean_count:.0f}', color='red', fontsize=9, fontweight='bold')

ax.set_title('Class Imbalance Overview', fontsize=13,
             fontweight='bold', color='#1a1a2e', pad=12)
ax.set_xlabel('Number of Images', fontsize=10)
ax.set_xlim(0, df_summary['Total'].max() * 1.2)
ax.set_facecolor('white')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
ax.legend(fontsize=9, frameon=False)

# Imbalance annotation
max_val = sorted_df['Total'].max()
min_val = sorted_df['Total'].min()
ax.annotate(
    f'Imbalance ratio: {max_val/min_val:.1f}×',
    xy=(max_val * 0.95, len(sorted_df) - 1),
    fontsize=9, color='#C44E52', fontweight='bold',
    ha='right'
)

plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class imbalance plot saved as class_imbalance.png')
print(f'\n⚠️  Imbalance ratio: {max_val/min_val:.1f}x')
print('   → Consider data augmentation or class weighting during model training.')

---
## 🖼️ Step 7 — Sample Images per Class

In [ ]:
def load_sample_image(folder_path, class_name, size=(180, 180)):
    """Load and resize one sample image from a class folder."""
    cls_dir = os.path.join(folder_path, class_name)
    if not os.path.exists(cls_dir):
        return None
    imgs = [
        f for f in os.listdir(cls_dir)
        if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS
    ]
    if not imgs:
        return None
    try:
        img = Image.open(os.path.join(cls_dir, imgs[0])).convert('RGB')
        return np.array(img.resize(size))
    except Exception:
        return None

# ── Build grid ────────────────────────────────────────────────────────────────
classes_available = [c for c in df_summary['Class']
                     if os.path.exists(os.path.join(TRAIN_PATH, c))]
n_classes = len(classes_available)
cols = 3
rows = -(-n_classes // cols)  # ceiling division

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4.2))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Sample Dermoscopic Images per Class',
             fontsize=15, fontweight='bold', color='white', y=1.01)

axes_flat = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

for i, cls in enumerate(classes_available):
    ax = axes_flat[i]
    img = load_sample_image(TRAIN_PATH, cls)
    if img is not None:
        ax.imshow(img)
    else:
        ax.set_facecolor('#2d3748')
        ax.text(0.5, 0.5, 'No Image Found',
                ha='center', va='center', color='#718096',
                fontsize=10, transform=ax.transAxes)
    count = df_summary.loc[df_summary['Class'] == cls, 'Total'].values[0]
    ax.set_title(f'{cls}\n({count} images)',
                 fontsize=9, fontweight='bold',
                 color=COLORS[i % len(COLORS)], pad=6)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(COLORS[i % len(COLORS)])
        spine.set_linewidth(1.5)
        spine.set_visible(True)

# Hide unused axes
for j in range(n_classes, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✅ Sample image grid saved as sample_images.png')

---
## 📋 Step 8 — Full EDA Dashboard (Combined)

Generates a single high-resolution dashboard combining all key plots.

In [ ]:
fig = plt.figure(figsize=(22, 26), facecolor='#f8f9fa')
fig.suptitle(
    '🔬 Skin Cancer ISIC — Exploratory Data Analysis Dashboard',
    fontsize=18, fontweight='bold', color='#1a1a2e', y=0.99
)
gs = gridspec.GridSpec(4, 3, figure=fig,
                       hspace=0.50, wspace=0.35,
                       top=0.95, bottom=0.04,
                       left=0.06, right=0.97)

def style(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor('white')
    for s in ax.spines.values(): s.set_color('#ddd')
    ax.tick_params(colors='#444', labelsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold',
                 color='#1a1a2e', pad=10)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9, color='#555')
    if ylabel: ax.set_ylabel(ylabel, fontsize=9, color='#555')
    ax.grid(axis='y', color='#eee', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

# Plot 1 — Class Distribution bar
ax1 = fig.add_subplot(gs[0, :2])
bars1 = ax1.bar(df_summary['Class'], df_summary['Total'],
                color=[class_colors[c] for c in df_summary['Class']],
                edgecolor='white', linewidth=0.8, zorder=3)
for bar, val in zip(bars1, df_summary['Total']):
    pct = val / df_summary['Total'].sum() * 100
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
             f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=7.5)
style(ax1, 'Total Image Count per Class', ylabel='Images')
ax1.set_xticklabels(df_summary['Class'], rotation=25, ha='right', fontsize=8)
ax1.set_ylim(0, df_summary['Total'].max() * 1.22)

# Plot 2 — Pie
ax2 = fig.add_subplot(gs[0, 2])
wedges, _, autotexts = ax2.pie(
    df_summary['Total'], labels=None, autopct='%1.1f%%',
    colors=[class_colors[c] for c in df_summary['Class']],
    startangle=140, pctdistance=0.78,
    wedgeprops=dict(edgecolor='white', linewidth=1.2)
)
for at in autotexts: at.set_fontsize(7); at.set_color('white')
ax2.set_title('Class Share', fontsize=11, fontweight='bold', color='#1a1a2e', pad=10)
ax2.legend(df_summary['Class'], loc='lower center',
           bbox_to_anchor=(0.5, -0.28), ncol=2, fontsize=7, frameon=False)

# Plot 3 — Grouped bar: Train vs Test
ax3 = fig.add_subplot(gs[1, :])
x, w = np.arange(len(df_summary)), 0.38
b1 = ax3.bar(x - w/2, df_summary['Train'], w, label='Train',
             color='#4C72B0', edgecolor='white', linewidth=0.7, zorder=3)
b2 = ax3.bar(x + w/2, df_summary['Test'],  w, label='Test',
             color='#DD8452', edgecolor='white', linewidth=0.7, zorder=3)
for bar, val in zip(list(b1)+list(b2), list(df_summary['Train'])+list(df_summary['Test'])):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontsize=7.5)
ax3.set_xticks(x)
ax3.set_xticklabels(df_summary['Class'], rotation=20, ha='right', fontsize=9)
style(ax3, 'Train vs Test Split per Class', ylabel='Images')
ax3.legend(fontsize=9, frameon=False)
ax3.set_ylim(0, max(df_summary['Train'].max(), df_summary['Test'].max()) * 1.18)

# Plot 4 — Imbalance horizontal bars
ax4 = fig.add_subplot(gs[2, :2])
s_df = df_summary.sort_values('Total')
ax4.barh(s_df['Class'], s_df['Total'],
         color=[class_colors[c] for c in s_df['Class']],
         edgecolor='white', height=0.6, zorder=3)
for i, (_, row) in enumerate(s_df.iterrows()):
    ax4.text(row['Total'] + 2, i, f"{row['Total']}",
             va='center', fontsize=9, color='#333')
mean_v = df_summary['Total'].mean()
ax4.axvline(mean_v, color='red', lw=1.5, linestyle='--', zorder=4)
ax4.text(mean_v + 2, len(s_df) - 0.6, f'Mean: {mean_v:.0f}',
         color='red', fontsize=8.5, fontweight='bold')
ax4.set_facecolor('white')
for sp in ax4.spines.values(): sp.set_color('#ddd')
ax4.tick_params(colors='#444', labelsize=9)
ax4.set_title('Class Imbalance Overview', fontsize=11,
              fontweight='bold', color='#1a1a2e', pad=10)
ax4.set_xlabel('Number of Images', fontsize=9, color='#555')
ax4.grid(axis='x', color='#eee', linewidth=0.8)
ax4.set_axisbelow(True)

# Plot 5 — Overall Train/Test pie
ax5 = fig.add_subplot(gs[2, 2])
t_train = df_summary['Train'].sum()
t_test  = df_summary['Test'].sum()
ax5.pie([t_train, t_test],
        labels=[f'Train\n{t_train:,}', f'Test\n{t_test:,}'],
        colors=['#4C72B0', '#DD8452'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2),
        textprops={'fontsize': 9})
ax5.set_title('Overall Train / Test', fontsize=11,
              fontweight='bold', color='#1a1a2e', pad=10)

# Plot 6 — Summary stats table
ax6 = fig.add_subplot(gs[3, :])
ax6.axis('off')
stats = [
    ['Total Images',       f'{df_summary["Total"].sum():,}'],
    ['Total Classes',      '9'],
    ['Train Images',       f'{t_train:,}  ({t_train/df_summary["Total"].sum()*100:.1f}%)'],
    ['Test Images',        f'{t_test:,}  ({t_test/df_summary["Total"].sum()*100:.1f}%)'],
    ['Most Common Class',  f'{df_summary.iloc[0]["Class"]}  ({df_summary.iloc[0]["Total"]} images)'],
    ['Least Common Class', f'{df_summary.iloc[-1]["Class"]}  ({df_summary.iloc[-1]["Total"]} images)'],
    ['Imbalance Ratio',    f'{df_summary["Total"].max()/df_summary["Total"].min():.1f}x'],
    ['Avg per Class',      f'{df_summary["Total"].mean():.0f} images'],
]
tbl = ax6.table(cellText=stats, colLabels=['Metric', 'Value'],
                cellLoc='left', loc='center',
                bbox=[0.05, 0.0, 0.9, 1.0])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor('#ddd')
    if row == 0:
        cell.set_facecolor('#1a1a2e'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f0f4fa')
    else:
        cell.set_facecolor('white')
ax6.set_title('Dataset Summary Statistics', fontsize=11,
              fontweight='bold', color='#1a1a2e', pad=10)

plt.savefig('skin_cancer_eda_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('✅ Full EDA dashboard saved as skin_cancer_eda_dashboard.png')

---
## 💾 Step 9 — Download All Outputs

In [ ]:
from google.colab import files

output_files = [
    'skin_cancer_eda_dashboard.png',
    'class_distribution.png',
    'train_test_split.png',
    'class_imbalance.png',
    'sample_images.png',
]

print('📥 Downloading output files...')
for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f'   ✅ {f}')
    else:
        print(f'   ⚠️  {f} not found — run the corresponding cell first')

print('\n🎉 All done!')

---
## 🔑 Key Findings

Based on the EDA performed above:

1. **Class Imbalance** — The dataset is imbalanced. Melanoma and Nevus dominate while Vascular Lesion and Dermatofibroma have significantly fewer images. Consider **data augmentation** or **class weighting** during model training.

2. **Train/Test Split** — The dataset maintains a consistent ~80/20 train/test split across most classes.

3. **Image Quality** — All images are standardised dermoscopic images from the ISIC archive, ensuring consistent imaging conditions.

4. **Clinical Importance** — Malignant classes (Melanoma, Basal Cell Carcinoma, Squamous Cell Carcinoma) are present in reasonable quantities for training a detection model.

---

## ➡️ Next Steps

- [ ] Data Augmentation (rotation, flip, zoom) to address class imbalance
- [ ] Build CNN model (EfficientNet / ResNet / MobileNet)
- [ ] Apply transfer learning with pretrained ImageNet weights
- [ ] Evaluate with confusion matrix, precision, recall, F1-score
- [ ] Deploy model for real-time dermoscopic image analysis

---
*Dataset: ISIC — International Skin Imaging Collaboration | License: CC0 Public Domain*